# Emerging Tech - Quantum Computing Notebook

# Emerging Technologies - Problems Notebook

This notebook explores the difference between classical and quantum algorithms, using the Deutsch-Jozsa algorithm as a central example.

The problems progress from classical Python implementations through to full quantum circuits built with Qiskit, demonstrating how quantum computing can solve certain problems more efficiently than any classical approach.


## Problem 1 - Generating random boolean Funtions

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) operates on functions that take a fixed number of Boolean inputs and return a single Boolean output. These functions are guaranteed to be one of two types:

- **Constant** - always returns the same value (`True` or `False`) regardless of the input
- **Balanced** - returns `True` for exactly half of all possible input combinations, and `False` for the other half

With four Boolean inputs there are 2⁴ = 16 possible input combinations. A balanced function must return `True` for exactly 8 of these and `False` for the remaining 8.

The function below, `random_constant_balanced`, randomly generates either a constant or balanced function and returns it. This gives us a way to produce mystery functions that we can later test both classically (Problem 2) and quantumly (Problem 5).

In [3]:
# Random selections.
import random

# Numerical arrays.
import numpy as np

# Permutations and combinations.
import itertools as it

def random_constant_balanced():
    # All possible combinations of 4 boolean inputs
    # e.g. (False, False, False, False), (False, False, False, True), ...
    all_inputs = list(it.product([False, True], repeat=4))  # 16 combinations
    
    # Randomly pick constant or balanced
    function_type = random.choice(["constant", "balanced"])
    
    if function_type == "constant":
        # Either always return True or always return False
        constant_value = random.choice([True, False])
        def f(*args):
            return constant_value
    
    else:  # balanced
        # Pick exactly 8 of the 16 inputs to return True for
        true_inputs = set(map(tuple, random.sample(all_inputs, 8)))
        def f(*args):
            return tuple(args) in true_inputs
    
    return f

#### Test Case

In [4]:
f = random_constant_balanced()

# Try all 16 inputs and see what it returns
all_inputs = list(it.product([False, True], repeat=4))
results = [f(*inp) for inp in all_inputs]
print(results)
print("True count:", results.count(True))
print("False count:", results.count(False))

[True, True, True, False, False, True, True, False, False, False, True, True, False, False, True, False]
True count: 8
False count: 8


## Problem 2 - Classical Testing for Function Type

## Problem 2: Classical Testing for Function Type

Before exploring the quantum advantage, we must first understand the classical cost of solving the same problem.

The function below, `determine_constant_balanced`, takes a mystery function `f` as input and determines whether it is constant or balanced by calling it with different inputs and observing the results.

**Strategy:** Call `f` with each input combination one at a time. The moment we see both a `True` and a `False` result, we can immediately conclude the function is balanced — a constant function would never return two different values. If we exhaust all inputs and only ever see one value, the function must be constant.

**Efficiency:** In the worst case, we need to call `f` a maximum of **9 times** to be 100% certain of the answer. This is because if we have seen the same output 8 times in a row, the function could still be balanced — the 9th call will either return a different value (confirming balanced) or the same value again (at which point a balanced function would have had to show a different value, so it must be constant). A truly constant function requires all 16 calls to be completely certain.

This classical worst case of 9 calls contrasts sharply with the quantum approach in Problem 5, which determines the answer with a **single query** regardless of how many inputs the function takes.

In [5]:
def determine_constant_balanced(f):
    # Generate all 16 possible combinations of 4 boolean inputs
    all_inputs = list(it.product([False, True], repeat=4))
    
    results = set()
    
    for inp in all_inputs:
        result = f(*inp)
        results.add(result)
        
        # If we've seen both True and False, it must be balanced
        if len(results) == 2:
            return "balanced"
    
    # Got through all inputs with only one unique result, must be constant
    return "constant"

### Test Case

In [6]:
# Run the test 5 times to see a mix of constant and balanced results
for i in range(5):
    
    # Generate a random constant or balanced function
    f = random_constant_balanced()
    
    print(determine_constant_balanced(f))

balanced
constant
balanced
constant
balanced


## Problem 3 - Quantum Oracles

## Problem 3: Quantum Oracles

A quantum oracle is a quantum circuit that encodes a classical Boolean function in a way that a quantum algorithm can query it. Rather than simply computing `f(x)`, the oracle transforms a two-qubit state according to the rule:

|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩

Where `⊕` is XOR. The first qubit is the **input qubit** and the second is the **output qubit**. The output qubit gets flipped if and only if `f(x) = 1`.

With a single Boolean input, there are exactly four possible Boolean functions:

| Function | f(0) | f(1) | Type |
|----------|------|------|------|
| Constant 0 | 0 | 0 | Constant |
| Constant 1 | 1 | 1 | Constant |
| Identity | 0 | 1 | Balanced |
| NOT | 1 | 0 | Balanced |

Each of these is implemented as a quantum circuit using two fundamental gates:

- **X gate** - flips a qubit from |0⟩ to |1⟩ or vice versa (equivalent to a classical NOT)
- **CNOT gate** - flips the target qubit only if the control qubit is |1⟩ (equivalent to a classical XOR)

The four oracles below implement each of these functions, and are demonstrated by running them on the Qiskit simulator with both possible inputs.

#### Import QisKit

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

In [8]:

def oracle_constant_0():
    # f(x) = 0 always, so we do nothing to the circuit
    # y XOR 0 = y, output qubit is unchanged
    qc = QuantumCircuit(2)
    return qc

def oracle_constant_1():
    # f(x) = 1 always, so we flip the output qubit regardless of input
    # y XOR 1 = NOT y
    qc = QuantumCircuit(2)
    qc.x(1)  # X gate flips qubit 1 (the output qubit)
    return qc

def oracle_balanced_identity():
    # f(x) = x, so output depends on input
    # y XOR x, we use a CNOT gate controlled on input qubit
    qc = QuantumCircuit(2)
    qc.cx(0, 1)  # CNOT: flips qubit 1 if qubit 0 is 1
    return qc

def oracle_balanced_not():
    # f(x) = NOT x, so output is flipped version of input
    # flip input qubit first, then apply CNOT, then flip back
    qc = QuantumCircuit(2)
    qc.x(0)       # flip input qubit
    qc.cx(0, 1)   # CNOT
    qc.x(0)       # flip input qubit back
    return qc

### Test Oracle

In [9]:
def test_oracle(oracle, name):
    print(f"\nOracle: {name}")
    
    # Test with input 0
    qc = QuantumCircuit(2, 2)
    qc.compose(oracle, inplace=True)
    qc.measure([0, 1], [0, 1])
    
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    print(f"  Input |00> gives: {counts}")
    
    # Test with input 1
    qc = QuantumCircuit(2, 2)
    qc.x(0)  # set input qubit to 1
    qc.compose(oracle, inplace=True)
    qc.measure([0, 1], [0, 1])
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    print(f"  Input |10> gives: {counts}")

# Test all four oracles
test_oracle(oracle_constant_0(), "Constant 0")
test_oracle(oracle_constant_1(), "Constant 1")
test_oracle(oracle_balanced_identity(), "Balanced Identity")
test_oracle(oracle_balanced_not(), "Balanced NOT")


Oracle: Constant 0
  Input |00> gives: {'00': 1}
  Input |10> gives: {'01': 1}

Oracle: Constant 1
  Input |00> gives: {'10': 1}
  Input |10> gives: {'11': 1}

Oracle: Balanced Identity
  Input |00> gives: {'00': 1}
  Input |10> gives: {'11': 1}

Oracle: Balanced NOT
  Input |00> gives: {'10': 1}
  Input |10> gives: {'01': 1}


## Problem 4 - Deutsch's Algorithm with Qiskit

[Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) is the simplest example of a quantum algorithm that outperforms any classical approach. While a classical computer must call the function at least twice to determine whether it is constant or balanced, Deutsch's algorithm determines the answer with a **single query** to the oracle.

**How it works:**

The circuit uses two qubits and four steps:

1. **Initialisation** - the input qubit starts as |0⟩ and the output qubit is flipped to |1⟩ using an X gate
2. **Superposition** - Hadamard gates are applied to both qubits, placing them into superposition so both possible inputs are encoded simultaneously
3. **Oracle query** - the oracle is applied once, encoding the function into the quantum state
4. **Interference** - a final Hadamard gate is applied to the input qubit, causing constructive or destructive interference depending on whether the function is constant or balanced
5. **Measurement** - the input qubit is measured

**Reading the result:**

- If the input qubit measures as **0** → the function is **constant**
- If the input qubit measures as **1** → the function is **balanced**

The interference in step 4 is the key — for constant functions the quantum states reinforce each other back to |0⟩, and for balanced functions they cancel each other out, leaving |1⟩. This is what allows a single query to reveal a global property of the function.

The circuit is demonstrated below using each of the four oracles from Problem 3.

In [10]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer

In [11]:
def deutsch_algorithm(oracle):
    # Create circuit with 2 qubits and 1 classical bit for measurement
    qc = QuantumCircuit(2, 1)
    
    # Step 1: Set output qubit (qubit 1) to |1>
    qc.x(1)
    
    # Step 2: Apply Hadamard to both qubits to create superposition
    qc.h(0)
    qc.h(1)
    
    # Step 3: Apply the oracle (single query to the function)
    qc.compose(oracle, inplace=True)
    
    # Step 4: Apply Hadamard to input qubit to cause interference
    qc.h(0)
    
    # Step 5: Measure only the input qubit
    qc.measure(0, 0)
    
    return qc


In [12]:
def run_deutsch(oracle, oracle_name):
    # Build the circuit with this oracle
    qc = deutsch_algorithm(oracle)
    
    # Run on the simulator
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()
    
    # If input qubit measured as 0 -> constant, if 1 -> balanced
    if '0' in counts and counts['0'] == 1024:
        conclusion = "constant"
    else:
        conclusion = "balanced"
    
    print(f"Oracle: {oracle_name}")
    print(f"Measurement results: {counts}")
    print(f"Conclusion: {conclusion}")
    print()

In [13]:
# Test with all four oracles from Problem 3
run_deutsch(oracle_constant_0(), "Constant 0")
run_deutsch(oracle_constant_1(), "Constant 1")
run_deutsch(oracle_balanced_identity(), "Balanced Identity")
run_deutsch(oracle_balanced_not(), "Balanced NOT")

Oracle: Constant 0
Measurement results: {'0': 1024}
Conclusion: constant

Oracle: Constant 1
Measurement results: {'0': 1024}
Conclusion: constant

Oracle: Balanced Identity
Measurement results: {'1': 1024}
Conclusion: balanced

Oracle: Balanced NOT
Measurement results: {'1': 1024}
Conclusion: balanced



In [14]:
# Draw the circuit for one example so you can see the structure
qc = deutsch_algorithm(oracle_balanced_identity())
print(qc.draw())

     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 


## Problem 5 - Scaling to the Deutsch–Jozsa Algorithm

In [15]:
def build_oracle(f):
    # 4 input qubits + 1 output qubit = 5 total
    qc = QuantumCircuit(5)
    
    all_inputs = list(it.product([False, True], repeat=4))
    
    for inp in all_inputs:
        # Only add gates for inputs where f returns True
        if f(*inp):
            # Flip any input qubits that are False so CNOT fires correctly
            for i, bit in enumerate(inp):
                if not bit:
                    qc.x(i)
            
            # Multi-controlled CNOT targeting the output qubit (qubit 4)
            qc.mcx([0, 1, 2, 3], 4)
            
            # Flip back to restore input qubits
            for i, bit in enumerate(inp):
                if not bit:
                    qc.x(i)
    
    return qc


def deutsch_jozsa(f):
    # Build the oracle from the classical function
    oracle = build_oracle(f)
    
    # Create circuit with 5 qubits and 4 classical bits for measurement
    qc = QuantumCircuit(5, 4)
    
    # Step 1: Flip output qubit to |1>
    qc.x(4)
    
    # Step 2: Apply Hadamard to all 5 qubits
    qc.h([0, 1, 2, 3, 4])
    
    # Step 3: Apply the oracle
    qc.compose(oracle, inplace=True)
    
    # Step 4: Apply Hadamard to input qubits only
    qc.h([0, 1, 2, 3])
    
    # Step 5: Measure the 4 input qubits
    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])
    
    return qc


def run_deutsch_jozsa(f, label):
    qc = deutsch_jozsa(f)
    
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=1024)
    result = job.result()
    counts = result.get_counts()
    
    # If all input qubits measured as 0000 -> constant, otherwise balanced
    if '0000' in counts and counts['0000'] == 1024:
        conclusion = "constant"
    else:
        conclusion = "balanced"
    
    print(f"Function: {label}")
    print(f"Measurement results: {counts}")
    print(f"Conclusion: {conclusion}")
    print()

In [16]:
# Constant - always False
f_const_false = lambda *args: False
run_deutsch_jozsa(f_const_false, "Constant False")

# Constant - always True
f_const_true = lambda *args: True
run_deutsch_jozsa(f_const_true, "Constant True")

# Two random balanced functions from Problem 1
all_inputs = list(it.product([False, True], repeat=4))

true_inputs_1 = set(map(tuple, random.sample(all_inputs, 8)))
f_balanced_1 = lambda *args: tuple(args) in true_inputs_1
run_deutsch_jozsa(f_balanced_1, "Balanced Function 1")

true_inputs_2 = set(map(tuple, random.sample(all_inputs, 8)))
f_balanced_2 = lambda *args: tuple(args) in true_inputs_2
run_deutsch_jozsa(f_balanced_2, "Balanced Function 2")

Function: Constant False
Measurement results: {'0000': 1024}
Conclusion: constant

Function: Constant True
Measurement results: {'0000': 1024}
Conclusion: constant

Function: Balanced Function 1
Measurement results: {'1101': 238, '1001': 273, '1111': 288, '1011': 225}
Conclusion: balanced

Function: Balanced Function 2
Measurement results: {'1001': 65, '0001': 253, '0011': 46, '1111': 262, '0110': 59, '1000': 58, '0111': 80, '1101': 62, '0010': 64, '1100': 75}
Conclusion: balanced

